# Task 1: Baseline hallucination detection with token probabilities

**Idea.** When a model is unsure, its probability is spread over many next tokens. So an answer made of low-probability
tokens (or high-entropy steps) might be a hallucination. We test this on TruthfulQA.

**Steps**
1. Qwen2.5-3B-Instruct answers all 817 questions with greedy decoding (always the most likely token).
2. During that same pass we keep, for every generated token, its log-probability and the entropy of the distribution.
   Score of an answer = mean log-probability and mean entropy over its tokens.
3. Each answer is labelled accurate or hallucinated by comparing it with TruthfulQA's reference answers.
4. AUC-PR with "hallucinated" as the positive class.

Runtime: T4 GPU, about 15 minutes. Outputs go to `MyDrive/project7/outputs/task1/`.

In [ ]:
# save everything to google drive so task 2 and task 3 can read it later
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/project7/outputs', exist_ok=True)
os.chdir('/content/drive/MyDrive/project7')

!pip install -q bert-score datasets

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

SEED = 42
MODEL = "Qwen/Qwen2.5-3B-Instruct"     # small open model, fits a free T4
SYSTEM = "Answer the question in one short sentence."
MAX_NEW = 64                            # answers are short, 64 tokens is plenty
BATCH = 16
OUT = "outputs/task1"
os.makedirs(OUT, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU"

## Data
TruthfulQA, `generation` config: 817 questions, each with lists of correct and incorrect reference answers.

In [ ]:
df = load_dataset("truthfulqa/truthful_qa", "generation", split="validation").to_pandas()
df["id"] = range(len(df))
print(len(df), "questions")
df[["question", "best_answer"]].head()

## Greedy answers + token log-probs and entropy
`output_logits=True` gives the raw logits at every generation step, so we do not need a second forward pass.
For each token: log p(token) and entropy H = -sum p log p. We average over the answer's tokens (everything before the
end-of-text token). The answers are saved to Drive, so running the notebook again skips this step.

In [ ]:
ans_file = f"{OUT}/answers.csv"

if os.path.exists(ans_file):
    res = pd.read_csv(ans_file)
    res["answer"] = res["answer"].fillna("")
    print("loaded saved answers")
else:
    tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16)
    except TypeError:  # older transformers calls it torch_dtype
        model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16)
    model = model.to("cuda").eval()

    eos = model.generation_config.eos_token_id
    eos = eos if isinstance(eos, list) else [eos]
    stop = torch.tensor(sorted(set(eos + [tok.pad_token_id])), device="cuda")

    @torch.no_grad()
    def answer(questions):
        prompts = [tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}],
                                           tokenize=False, add_generation_prompt=True) for q in questions]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        out = model.generate(**enc, do_sample=False, max_new_tokens=MAX_NEW, repetition_penalty=1.0,
                             temperature=None, top_p=None, top_k=None, pad_token_id=tok.pad_token_id,
                             return_dict_in_generate=True, output_logits=True)
        gen = out.sequences[:, enc["input_ids"].shape[1]:]                    # only the new tokens
        logp = torch.log_softmax(torch.stack(out.logits, dim=1).float(), dim=-1)
        tok_logp = logp.gather(-1, gen.unsqueeze(-1)).squeeze(-1)             # log p of the chosen token
        entropy = -(logp.exp() * logp).sum(-1)                                # entropy at each step
        keep = (torch.isin(gen, stop).cumsum(1) == 0).float()                 # tokens before the first end token
        n = keep.sum(1)
        mean_lp = ((tok_logp * keep).sum(1) / n).tolist()                     # nan if the answer is empty
        mean_ent = ((entropy * keep).sum(1) / n).tolist()
        texts = [t.strip() for t in tok.batch_decode(gen, skip_special_tokens=True)]
        return texts, mean_lp, mean_ent, n.int().tolist()

    rows = []
    for b in range(0, len(df), BATCH):
        part = df.iloc[b:b + BATCH]
        texts, lp, ent, n = answer(part["question"].tolist())
        rows += [dict(id=i, answer=t, mean_logprob=l, mean_entropy=e, n_tokens=k)
                 for i, t, l, e, k in zip(part["id"], texts, lp, ent, n)]
        print(f"{b + len(part)}/{len(df)}", end="\r")
    res = pd.DataFrame(rows)
    res.to_csv(ans_file, index=False)
    del model
    torch.cuda.empty_cache()

res = df[["id", "question", "best_answer", "category"]].merge(res, on="id")
res[["question", "answer", "mean_logprob", "mean_entropy"]].head()

## Labels: is the answer accurate?
The dataset document says: accurate if the answer matches or closely paraphrases a correct reference, inaccurate otherwise.
We follow the TruthfulQA paper's similarity approach: compute the answer's best BERTScore-F1 against the correct references
and against the incorrect references. **Accurate if it is closer to a correct one than to any incorrect one.**
BERTScore uses `roberta-large`, the same model Task 2 needs, so no extra model is added.

This is automatic and therefore noisy (for example, "No, X" and "Yes, X" look alike to BERTScore). A small hand check
at the end of this notebook measures how often it is right.

In [ ]:
from bert_score import BERTScorer
scorer = BERTScorer(model_type="roberta-large", lang="en", rescale_with_baseline=True, device="cuda")

def best_f1(answers, ref_lists):
    # best BERTScore-F1 of each answer against its list of references
    cands, refs, owner = [], [], []
    for k, (a, rs) in enumerate(zip(answers, ref_lists)):
        for r in rs:
            cands.append(a); refs.append(r); owner.append(k)
    f1 = scorer.score(cands, refs, batch_size=64)[2].numpy()
    best = np.full(len(answers), -np.inf)
    np.maximum.at(best, owner, f1)
    return best

refs = df.set_index("id").loc[res["id"]]
res["sim_correct"] = best_f1(res["answer"].tolist(), refs["correct_answers"].tolist())
res["sim_incorrect"] = best_f1(res["answer"].tolist(), refs["incorrect_answers"].tolist())
res["hallucinated"] = (res["sim_correct"] <= res["sim_incorrect"]).astype(int)
print(f"hallucinated: {res.hallucinated.mean():.1%} of {len(res)} answers")

## Results: AUC-PR
Positive class = hallucinated. A random detector gets AUC-PR equal to the share of hallucinated answers, so that is the
baseline to beat. Lower log-probability and higher entropy should mean "more likely hallucinated", so we use
`-mean_logprob` and `mean_entropy` as the scores.

In [ ]:
ev = res[res["n_tokens"] > 0]          # an empty answer has no tokens to score
print("empty answers dropped:", len(res) - len(ev))
y = ev["hallucinated"].values
scores = {"-mean log-prob": -ev["mean_logprob"], "mean entropy": ev["mean_entropy"]}

table = pd.DataFrame({name: {"AUC-PR": average_precision_score(y, s), "AUROC": roc_auc_score(y, s)}
                      for name, s in scores.items()}).T
table.loc["random"] = [y.mean(), 0.5]
table.to_csv(f"{OUT}/task1_results.csv")
print(table.round(3))

for name, s in scores.items():
    p, r, _ = precision_recall_curve(y, s)
    plt.plot(r, p, label=name)
plt.axhline(y.mean(), ls="--", c="gray", label="random")
plt.xlabel("recall"); plt.ylabel("precision"); plt.title("Task 1 on TruthfulQA"); plt.legend()
plt.savefig(f"{OUT}/task1_pr.png", dpi=150); plt.show()

res.to_csv(f"{OUT}/task1_features.csv", index=False)

## Hand check of the labels (about 20 minutes, by you)
1. Running this cell writes `outputs/task1/hand_check.csv`: 50 random answers with their reference answers.
2. Open it in Google Sheets, fill `my_label` (1 = matches a correct reference, 0 = wrong), and save it as
   `hand_check_filled.csv` in the same folder.
3. Run this cell again: it prints how often the automatic label agrees with you. Put that number in the report.

In [ ]:
filled = f"{OUT}/hand_check_filled.csv"
if os.path.exists(filled):
    h = pd.read_csv(filled).dropna(subset=["my_label"])
    m = h[["id", "my_label"]].merge(res[["id", "hallucinated"]], on="id")
    agree = (m["my_label"] == 1 - m["hallucinated"]).mean()
    print(f"automatic label agrees with my label on {agree:.0%} of {len(m)} answers")
else:
    s = res.sample(50, random_state=SEED)
    ref = df.set_index("id")
    sheet = pd.DataFrame({"id": s["id"], "question": s["question"], "answer": s["answer"],
                          "correct_answers": [" | ".join(ref.loc[i, "correct_answers"]) for i in s["id"]],
                          "incorrect_answers": [" | ".join(ref.loc[i, "incorrect_answers"]) for i in s["id"]],
                          "my_label": ""})
    sheet.to_csv(f"{OUT}/hand_check.csv", index=False)
    print("wrote", f"{OUT}/hand_check.csv")

In [ ]:
# versions used in this run, for requirements.txt
import importlib.metadata as im
pkgs = ["torch", "transformers", "datasets", "bert-score", "scikit-learn", "numpy", "pandas", "matplotlib"]
lines = [f"{p}=={im.version(p).split('+')[0]}" for p in pkgs]
open("outputs/requirements.txt", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))